# Planet NDVI Increase Validation (Tile Mosaics)

This notebook mosaics PlanetScope SR tiles from `jamaica_planet_ndvi_validation/PSScene` into one image per acquisition date.

Expected result from your current download:
- `2025-08-30`: 4 tiles merged into 1 image
- `2025-11-18`: 2 tiles merged into 1 image

Output files are written to:
- `dphil_papers/dphil_paper_3/inputs/ndvi/jamaica_planet_ndvi_validation/mosaics_by_date`


In [ ]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.merge import merge

plt.style.use("default")


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "dphil_papers").exists():
            return p
    raise FileNotFoundError(f"Could not find project root from {start}")


ROOT = find_project_root(Path.cwd())
scene_dir = ROOT / "dphil_papers/dphil_paper_3/inputs/ndvi/jamaica_planet_ndvi_validation/PSScene"
output_dir = ROOT / "dphil_papers/dphil_paper_3/inputs/ndvi/jamaica_planet_ndvi_validation/mosaics_by_date"
output_dir.mkdir(parents=True, exist_ok=True)

tile_files = sorted(scene_dir.glob("*_3B_AnalyticMS_SR_clip.tif"))
if not tile_files:
    raise FileNotFoundError(f"No Planet SR clip tiles found in: {scene_dir}")

print("Project root:", ROOT)
print("Scene directory:", scene_dir)
print("Output directory:", output_dir)
print("Tile count:", len(tile_files))


In [ ]:
tiles_by_date = defaultdict(list)
for tif in tile_files:
    date_key = tif.name[:8]  # YYYYMMDD from filename prefix
    tiles_by_date[date_key].append(tif)

print("Tiles grouped by date:")
for date_key, files in sorted(tiles_by_date.items()):
    pretty_date = datetime.strptime(date_key, "%Y%m%d").strftime("%Y-%m-%d")
    print(f"  {pretty_date}: {len(files)} tile(s)")
    for f in files:
        print("   -", f.name)


In [ ]:
def mosaic_tiles(tiles: list[Path], out_path: Path, method: str = "first") -> Path:
    """Merge tiled Planet SR scenes into one raster for a given date."""
    srcs = [rasterio.open(tile) for tile in tiles]
    try:
        mosaic, transform = merge(srcs, method=method, nodata=0)

        profile = srcs[0].profile.copy()
        profile.update(
            {
                "driver": "GTiff",
                "height": mosaic.shape[1],
                "width": mosaic.shape[2],
                "transform": transform,
                "count": mosaic.shape[0],
                "compress": "deflate",
                "tiled": True,
                "blockxsize": 512,
                "blockysize": 512,
            }
        )

        if np.issubdtype(mosaic.dtype, np.integer):
            profile["predictor"] = 2
        elif np.issubdtype(mosaic.dtype, np.floating):
            profile["predictor"] = 3

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(mosaic)
    finally:
        for src in srcs:
            src.close()

    return out_path


In [ ]:
# Merge strategy for overlap: "first", "last", "min", "max"
MERGE_METHOD = "first"

mosaics_by_date = {}
for date_key, files in sorted(tiles_by_date.items()):
    pretty_date = datetime.strptime(date_key, "%Y%m%d").strftime("%Y-%m-%d")
    out_path = output_dir / f"planet_sr_mosaic_{pretty_date}.tif"
    mosaic_tiles(files, out_path, method=MERGE_METHOD)
    mosaics_by_date[pretty_date] = out_path

print("Wrote mosaics:")
for date_key, out_path in mosaics_by_date.items():
    print(f"  {date_key}: {out_path}")


In [ ]:
def quick_rgb_preview(path: Path, stretch=(2, 98)):
    """Quick visual check using Planet SR bands (R,G,B = 3,2,1)."""
    with rasterio.open(path) as src:
        blue = src.read(1).astype("float32")
        green = src.read(2).astype("float32")
        red = src.read(3).astype("float32")
        nodata = 0 if src.nodata is None else src.nodata

    rgb = np.dstack([red, green, blue])
    valid = np.all(rgb != nodata, axis=2)

    if np.any(valid):
        lo, hi = np.percentile(rgb[valid], stretch)
        rgb = (rgb - lo) / (hi - lo + 1e-6)
        rgb = np.clip(rgb, 0, 1)

    rgb[~valid] = np.nan
    return rgb


n_cols = max(1, len(mosaics_by_date))
fig, axes = plt.subplots(1, n_cols, figsize=(7 * n_cols, 7), squeeze=False)

for ax, (date_key, raster_path) in zip(axes[0], sorted(mosaics_by_date.items())):
    ax.imshow(quick_rgb_preview(raster_path))
    ax.set_title(date_key)
    ax.axis("off")

plt.tight_layout()
plt.show()
